# GRPO Training on MATH500 with TRL

Uses HuggingFace TRL's `GRPOTrainer` to apply reinforcement learning (GRPO) to Qwen3-0.6B on math problems.  
Reward signal: correctness (1.0/0.0) + format (0.5/0.0 for `<think>` tags).

In [1]:
!uv pip install trl transformers accelerate peft datasets sympy wandb

Using Python 3.10.18 environment at: /Users/yongsingyou/code/reasoning-from-scratch/.venv
Checked 7 packages in 47ms


In [2]:
from pathlib import Path
import urllib.request

TRAIN_PATH = Path("math_train.json")
if not TRAIN_PATH.exists():
    print("Downloading math_train.json ...")
    url = "https://raw.githubusercontent.com/rasbt/math_full_minus_math500/refs/heads/main/math_full_minus_math500.json"
    urllib.request.urlretrieve(url, TRAIN_PATH)
    print(f"Saved to {TRAIN_PATH} ({TRAIN_PATH.stat().st_size / 1e6:.1f} MB)")
else:
    print(f"math_train.json already exists ({TRAIN_PATH.stat().st_size / 1e6:.1f} MB)")

math_train.json already exists (11.0 MB)


In [3]:
import json
import os
import re
import torch
import wandb
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import GRPOTrainer, GRPOConfig
from peft import LoraConfig

# wandb reads WANDB_API_KEY; fall back to the name used in .zshrc
if not os.environ.get("WANDB_API_KEY"):
    os.environ["WANDB_API_KEY"] = os.environ.get("wandb_apikey", "")

if torch.cuda.is_available():
    device = "cuda"
    dtype = torch.bfloat16
elif torch.backends.mps.is_available():
    device = "mps"
    dtype = torch.float16  # MPS does not support bfloat16
else:
    device = "cpu"
    dtype = torch.float32

print(f"device: {device}, dtype: {dtype}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

/Users/yongsingyou/code/reasoning-from-scratch/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: mps, dtype: torch.float16


In [4]:
with open(TRAIN_PATH) as f:
    data = json.load(f)

SYSTEM_PROMPT = (
    "You are a math expert. Show your reasoning inside <think>...</think> tags, "
    "then give the final answer inside \\boxed{}."
)


def make_prompt(problem):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": problem},
    ]


subset = data if device == "cuda" else data[:64]
dataset = Dataset.from_list([{"prompt": make_prompt(ex["problem"]), "answer": ex["answer"]} for ex in subset])

print(f"Training dataset size: {len(dataset)} ({'full' if device == 'cuda' else 'local subset'})")
print(f"Example answer: {dataset[0]['answer']}")

Training dataset size: 64 (local subset)
Example answer: 0


In [5]:
model_id = "Qwen/Qwen3-0.6B"

tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# device_map="auto" does not work on MPS; load to CPU then move
if device == "cuda":
    model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=dtype, device_map="auto")
else:
    model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=dtype).to(device)

print(f"Model on {device}: {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M params")

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 311/311 [00:01<00:00, 239.05it/s]


Model on mps: 596M params


In [6]:
# TRL passes extra dataset columns as kwargs, so `answer` arrives automatically.


def extract_boxed(text):
    matches = re.findall(r"\\boxed\{([^}]*)\}", text)
    return matches[-1].strip() if matches else None


def reward_correctness(prompts, completions, answer, **kwargs):
    scores = []
    for completion, gt in zip(completions, answer):
        text = completion[0]["content"] if isinstance(completion, list) else completion
        extracted = extract_boxed(text)
        scores.append(1.0 if extracted is not None and extracted == gt.strip() else 0.0)
    return scores


def reward_format(prompts, completions, **kwargs):
    scores = []
    for completion in completions:
        text = completion[0]["content"] if isinstance(completion, list) else completion
        has_think = bool(re.search(r"<think>.*?</think>", text, re.DOTALL))
        scores.append(0.5 if has_think else 0.0)
    return scores


# Sanity check
test_completions = [
    "<think>Let me solve this.</think> The answer is \\boxed{42}",
    "The answer is \\boxed{7}",
    "I don't know.",
]
test_answers = ["42", "99", "7"]
print("correctness:", reward_correctness(None, test_completions, test_answers))
print("format:     ", reward_format(None, test_completions))

correctness: [1.0, 0.0, 0.0]
format:      [0.5, 0.0, 0.0]


In [15]:
# Scale down for MPS (local testing); full settings for CUDA (Colab/Kaggle)
is_cuda = device == "cuda"

# Remove stale wandb Jupyter hooks before re-initializing
try:
    ip = get_ipython()
    for event in ["pre_run_cell", "post_run_cell"]:
        ip.events.callbacks[event] = [
            cb for cb in ip.events.callbacks.get(event, [])
            if "wandb" not in str(cb)
        ]
except Exception:
    pass

wandb.finish()
wandb.init(
    project="grpo-qwen3-math",
    config={
        "model": "Qwen/Qwen3-0.6B",
        "device": device,
        "num_generations": 4 if is_cuda else 2,
        "max_completion_length": 512 if is_cuda else 128,
        "learning_rate": 1e-6,
        "beta": 0.01,
    },
)

config = GRPOConfig(
    output_dir="grpo-qwen3-math",
    num_generations=4 if is_cuda else 2,
    max_completion_length=512 if is_cuda else 128,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8 if is_cuda else 2,
    learning_rate=1e-6,
    num_train_epochs=1,
    max_steps=-1 if is_cuda else 10,
    beta=0.01,
    bf16=is_cuda,
    fp16=False,
    logging_steps=1,
    save_steps=50 if is_cuda else 5,
    save_total_limit=3,
    save_only_model=True,
    report_to="wandb",
)

Error in callback <bound method _WandbInit._pre_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x12c39e5f0>> (for pre_run_cell), with arguments args (<ExecutionInfo object at 1430b7280, raw_cell="# Scale down for MPS (local testing); full setting.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell:/Users/yongsingyou/code/reasoning-from-scratch/learn/train.ipynb#X55sZmlsZQ%3D%3D>,),kwargs {}:


ConnectionResetError: Connection lost

ConnectionResetError: Connection lost

In [8]:
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
)

In [ ]:
trainer = GRPOTrainer(
    model=model,
    args=config,
    train_dataset=dataset,
    reward_funcs=[reward_correctness, reward_format],
    peft_config=peft_config,
    processing_class=tokenizer,
)

trainer.train()
trainer.save_model("grpo-qwen3-math/final")

In [16]:
model.eval()
n_eval = 2
correct = 0

for ex in data[:n_eval]:
    messages = make_prompt(ex["problem"])
    encoded = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt", return_dict=True
    )
    input_ids = encoded["input_ids"].to(model.device)
    attention_mask = encoded["attention_mask"].to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            attention_mask=attention_mask,
            max_new_tokens=256,
            temperature=0.0,
            do_sample=False,
        )

    completion = tokenizer.decode(output_ids[0][input_ids.shape[1] :], skip_special_tokens=True)
    extracted = extract_boxed(completion)
    if extracted and extracted == ex["answer"].strip():
        correct += 1

accuracy = correct / n_eval
wandb.log({"eval/accuracy": accuracy, "eval/n": n_eval})
wandb.finish()

print(f"Accuracy on first {n_eval} problems: {correct}/{n_eval} = {accuracy:.1%}")

ConnectionResetError: Connection lost

In [ ]:
# from peft import PeftModel

# model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=dtype).to(device)
# model = PeftModel.from_pretrained(model, "grpo-qwen3-math/final")